# ICS Anomaly Detection Benchmark

UREx CDE2605R — benchmarking XGBoost against LSTM sequence models for anomaly
detection in Industrial Control Systems.

This notebook is a runnable, annotated version of `src/experiment.py`, one
dataset at a time. Set `DATASET` below and re-run — everything downstream
(loading, splitting, training, evaluation) reuses the project's `src/`
modules directly (`config.py`, `data.py`, `models.py`, `evaluate.py`,
`experiment.py`), not reimplemented logic, so results here are identical to
running the CLI on the same dataset.

**The project's four methodological commitments** (see `README.md`), all
enforced by the code this notebook calls, regardless of which dataset is
selected:
1. Every metric is reported under both the strict *pointwise* protocol and the
   inflationary *point-adjust* protocol used by most published ICS papers.
2. Splits are chronological — no shuffling, no `random_state`.
3. Scalers are fit on training data only.
4. Thresholds are chosen on validation, applied unchanged to test.


## Setup

The two environment variables below must be set **before** XGBoost or PyTorch
are imported anywhere in this process. Both libraries bundle their own copy of
the OpenMP runtime (`libomp.dylib`), and on macOS, loading both into one
process without this can deadlock silently — the kernel just hangs with 0%
CPU, no error, no traceback. This was hit and diagnosed during CLI runs of
this same pipeline.


In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path

# This notebook lives in notebooks/; the project modules live in ../src.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
assert SRC.exists(), f"expected {SRC} to exist — run this notebook from notebooks/"
sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd

print(f"project root: {ROOT}")


project root: /Users/santhosh/Desktop/urex-ics


## Choose a dataset

`DATASET` must be a key in `data.LOADERS` (== `config.DATASET_INFO`). Change
this one line and re-run the notebook to benchmark a different dataset — no
other cell needs editing.

Not every registered dataset has a working loader yet — `src/data.py` ships
`load_wadi`, `load_morris`, and `load_batadal` as stubs that raise
`NotImplementedError` until each dataset's files arrive and its loader is
filled in (see `notes/datasets.md` for status). `load_swat` is currently the
only one implemented — this project's scope is SWaT only for now.


In [2]:
from data import LOADERS

DATASET = "swat"  # <-- change this, then Run All

print(f"registered: {list(LOADERS)}")
print(f"running:    {DATASET!r}")

registered: ['hai', 'swat', 'wadi', 'morris', 'batadal']
running:    'swat'


## Load the dataset

`data.load(name)` dispatches to that dataset's loader (`load_swat()`: reads
the two raw iTrust xlsx files, builds the `label` column, handles SWaT's
specific cleanup) and then applies `chronological_split()` — 60% train / 20%
val / 20% test targets, in time order, nudged to never bisect an attack
episode, scaler fit on train only. The split logic and its guardrails are
shared by every dataset in `src/data.py`; only the loader itself is
dataset-specific.


In [3]:
from config import RESULTS, THRESHOLD_PERCENTILE, DATASET_INFO
from data import load

info = DATASET_INFO.get(DATASET, {})
print(f"{DATASET}: {info}")
print()

split = load(DATASET)
print(split.summary())


swat: {'source': 'iTrust, SUTD', 'domain': 'water treatment', 'temporal_dependency': 'high', 'role': 'primary, literature comparability'}



/Users/santhosh/.venv/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/santhosh/.venv/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


swat: 51 features
  train  555,071 rows  anomaly rate 0.0093
  val    185,024 rows  anomaly rate 0.2295
  test   185,024 rows  anomaly rate 0.0376


In [4]:
print(f"features: {split.n_features}")
print(f"first 10 feature names: {split.feature_names[:10]}")


features: 51
first 10 feature names: ['FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201']


## The detectors

Every detector shares one interface — `detector.fit(split)` then
`detector.score(X)` — defined in `src/models.py`. That uniformity is what
lets the same evaluation code run over every model, on every dataset, without
special cases.

| name | family | supervised? | what the score measures |
|---|---|---|---|
| `random` | baseline | – | pure noise; the sanity floor every real model must beat |
| `xgboost` | gradient boosting | yes | each row scored independently, no explicit memory of the past |
| `xgboost_lags` | gradient boosting | yes | same, plus engineered lag/rolling/diff features so it can approximate short-term memory |
| `lstm_ae` | deep (autoencoder) | no (trained on normal rows only) | reconstruction error over a 60-second window — measures *compressibility* |
| `lstm_forecast` | deep (forecaster) | no (trained on normal rows only) | one-step prediction error — measures *predictability*, sees temporal structure the autoencoder cannot |
| `naive_bayes` | classical/shallow baseline | yes | per-feature Gaussian likelihood ratio |
| `random_forest` | classical/shallow baseline | yes | majority vote over bagged decision trees |
| `oner` | classical/shallow baseline | yes | one hand-picked feature, binned; the simplest rule that could work |
| `j48` | classical/shallow baseline | yes | single decision tree (C4.5 stand-in) |
| `nnge_stub_knn` | classical/shallow baseline | yes | k-NN vote (stand-in for NNge's merged exemplars) |
| `svm` | classical/shallow baseline | yes | RBF-kernel SVM via SMO, trained on a stratified subsample |

The deep family is deliberately represented by **two** models. Reporting only
the autoencoder would misattribute a scoring-rule limitation to deep learning
as a whole — see `LSTMAutoencoderDetector`'s docstring in `models.py` for the
full argument.

The six classical/shallow models round out the traditional-ML side of the
benchmark alongside `xgboost` — cheap, well-understood techniques (a
probabilistic classifier, bagged trees, a single tree, a rule baseline, an
instance-based method, and a kernel method) evaluated on this notebook's
chronological split, same threshold rule as everything else. See the
"classical / shallow supervised baselines" section of `models.py` for exactly
which are faithful implementations and which are deliberate stand-ins (`j48`
for C4.5, `nnge_stub_knn` for NNge's merged-exemplar matching).


In [5]:
from models import REGISTRY

MODEL_NAMES = [
    "random", "xgboost", "xgboost_lags", "lstm_ae", "lstm_forecast",
    "naive_bayes", "random_forest", "oner", "j48", "nnge_stub_knn", "svm",
]
print("registered models:", list(REGISTRY))

registered models: ['random', 'xgboost', 'xgboost_lags', 'lstm_ae', 'lstm_forecast', 'naive_bayes', 'random_forest', 'oner', 'j48', 'nnge_stub_knn', 'svm']


## Evaluation protocol: pointwise vs point-adjust

This is the project's central methodological point (`src/evaluate.py`), and
it applies identically regardless of which dataset is loaded above.

- **Pointwise** — each second judged independently. Strict, and the honest
  number to read first.
- **Point-adjust** — if a model flags *any one second* inside a ground-truth
  attack window, the *entire window* counts as detected. This is the
  dominant protocol in published ICS anomaly detection work, and Kim et al.
  (AAAI 2022) showed it lets a random score beat published state of the art.

`random_baseline_check()` reproduces that result directly: pure noise scored
against synthetic anomaly segments. If point-adjusted F1 comes back high
while pointwise F1 stays near zero, the metric implementation is correct —
and the literature's typical benchmark numbers are exposed as an artefact
of the protocol, not evidence the model actually works. This check does not
depend on `DATASET` — it runs on synthetic data every time as a fixed sanity
check.


In [6]:
from evaluate import random_baseline_check

sanity = random_baseline_check()
print(sanity.summary())


random noise on synthetic
  anomaly rate     0.0953   threshold 0.9903
  ROC-AUC          0.4979    PR-AUC 0.0948
  pointwise      P 0.108  R 0.011  F1 0.021  MCC 0.004
  point-adjusted P 0.910  R 0.945  F1 0.927  MCC 0.919
  F1 inflation     +0.906


## Run the benchmark

`experiment.run()` fits each detector on the training window, picks a
threshold at the 97th percentile of *validation* scores (never test), scores
the test window, evaluates under both protocols, prints a summary per model,
and saves the full results as JSON under `results/` — named from `DATASET`,
so runs against different datasets never overwrite each other.

The two LSTM models are the slow part (8 and 20 training epochs respectively)
— this cell's runtime scales with how much training data `DATASET` has.


In [7]:
from experiment import run

results = run(split, MODEL_NAMES, tag="notebook")


swat: 51 features
  train  555,071 rows  anomaly rate 0.0093
  val    185,024 rows  anomaly rate 0.2295
  test   185,024 rows  anomaly rate 0.0376

  random ...
random on swat
  anomaly rate     0.0376   threshold 0.9699
  ROC-AUC          0.4974    PR-AUC 0.0376
  pointwise      P 0.037  R 0.030  F1 0.033  MCC -0.001
  point-adjusted P 0.565  R 1.000  F1 0.722  MCC 0.740
  F1 inflation     +0.689

  xgboost ...
xgboost on swat
  anomaly rate     0.0376   threshold 0.6468
  ROC-AUC          0.6010    PR-AUC 0.0717
  pointwise      P 0.000  R 0.000  F1 0.000  MCC 0.000
  point-adjusted P 0.000  R 0.000  F1 0.000  MCC 0.000
  F1 inflation     +0.000

  xgboost+lags[1, 5, 30] ...
xgboost+lags[1, 5, 30] on swat
  anomaly rate     0.0376   threshold 0.5945
  ROC-AUC          0.5618    PR-AUC 0.0594
  pointwise      P 0.000  R 0.000  F1 0.000  MCC 0.000
  point-adjusted P 0.000  R 0.000  F1 0.000  MCC 0.000
  F1 inflation     +0.000

  lstm_ae(w=60,h=64) ...
    epoch   0  loss 0.08696
    e

/Users/santhosh/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm on swat
  anomaly rate     0.0376   threshold 0.2383
  ROC-AUC          0.6031    PR-AUC 0.0459
  pointwise      P 0.000  R 0.000  F1 0.000  MCC 0.000
  point-adjusted P 0.000  R 0.000  F1 0.000  MCC 0.000
  F1 inflation     +0.000

saved -> results/swat-notebook-20260920-105902.json

model                        PR-AUC    F1 pw   F1 adj  inflation
----------------------------------------------------------------
random                        0.038    0.033    0.722     +0.689
xgboost                       0.072    0.000    0.000     +0.000
xgboost+lags[1, 5, 30]        0.059    0.000    0.000     +0.000
lstm_ae(w=60,h=64)            0.451    0.000    0.000     +0.000
lstm_forecast(w=60,h=64)      0.490    0.000    0.000     +0.000
naive_bayes                   0.323    0.476    0.509     +0.033
random_forest                 0.085    0.000    0.000     +0.000
oner                          0.037    0.057    0.442     +0.385
j48                           0.038    0.056    0.284     +0

## Results, tidied

Same numbers as the printed table above, reshaped into a DataFrame for
easier reading/plotting. `f1_inflation` is `f1_point_adjust - f1_pointwise`
— the headline number. Large inflation means a model is close to guessing
and getting lucky once per attack window; read the pointwise column first.


In [8]:
summary = pd.DataFrame([
    {
        "model": r.meta.get("model"),
        "roc_auc": r.roc_auc,
        "pr_auc": r.pr_auc,
        "f1_pointwise": r.pointwise.f1,
        "f1_point_adjust": r.point_adjusted.f1,
        "f1_inflation": r.f1_inflation,
    }
    for r in results
]).set_index("model")

summary


,roc_auc,pr_auc,f1_pointwise,f1_point_adjust,f1_inflation
model,,,,,
random,0.497445,0.037568,0.032889,0.722219,0.689330
xgboost,0.600965,0.071699,0.000000,0.000000,0.000000
"xgboost+lags[1, 5, 30]",0.561850,0.059376,0.000000,0.000000,0.000000
"lstm_ae(w=60,h=64)",0.764395,0.451409,0.000000,0.000000,0.000000
"lstm_forecast(w=60,h=64)",0.800809,0.489591,0.000000,0.000000,0.000000
naive_bayes,0.651167,0.322673,0.476373,0.509133,0.032759
random_forest,0.703675,0.084939,0.000000,0.000000,0.000000
oner,0.487136,0.037373,0.056987,0.441954,0.384966
j48,0.502681,0.037844,0.055933,0.284388,0.228455


## How to read this

- **`random`'s `f1_inflation`** should be large and positive — that's the
  Kim et al. effect confirmed on real attack data, not just the synthetic
  sanity check above. If any real model's inflation is comparably large,
  its point-adjusted score is not trustworthy evidence it works.
- **Deep vs boosting**: check `DATASET_INFO[DATASET]["temporal_dependency"]`
  (printed when the dataset loaded above) against what `f1_pointwise` shows.
  The project's hypothesis is that sequence models (`lstm_ae`,
  `lstm_forecast`) have a structural advantage on high-temporal-dependency
  datasets over row-independent gradient boosting (`xgboost`); low-dependency
  datasets (Morris, once its loader exists) are the contrast case where
  boosting is expected to hold its own or win.
- **`xgboost` vs `xgboost_lags`**: whether engineered lag features close the
  gap with the deep models, widen it, or make boosting worse — all three
  are informative outcomes for the project's central question (how much of
  deep learning's advantage is architectural vs recoverable with feature
  engineering). Worth comparing across datasets, not just within one.
- Full precision/recall/MCC per model, per protocol, is in the saved JSON
  under `results/<DATASET>-notebook-<timestamp>.json` (path printed above).
